# Single-Cell RNA-seq Analysis: Pediatric B-ALL (GSE132509)
### Leukemic blasts vs. healthy bone marrow — single-patient pilot, then multi-patient replication

This notebook analyzes public pediatric acute lymphoblastic leukemia (B-ALL, ETV6-RUNX1 subtype) single-cell RNA-seq data against healthy pediatric bone marrow controls, from GEO series **GSE132509** (Caron et al., *Scientific Reports* 2020).

**Structure:**
- **Part A** — a quick single-patient pilot (1 leukemia sample vs. 1 healthy sample) to validate the pipeline end-to-end
- **Part B** — the full multi-patient analysis (4 leukemia patients vs. 3 healthy donors), which is the statistically meaningful comparison

**Data source:** GEO accession GSE132509. Samples used:
| GSM | Sample | Condition |
|---|---|---|
| GSM3872434 | ETV6-RUNX1_1 | Leukemia |
| GSM3872435 | ETV6-RUNX1_2 | Leukemia |
| GSM3872436 | ETV6-RUNX1_3 | Leukemia |
| GSM3872437 | ETV6-RUNX1_4 | Leukemia |
| GSM3872442 | PBMMC_1 | Healthy |
| GSM3872443 | PBMMC_2 | Healthy |
| GSM3872444 | PBMMC_3 | Healthy |

**Prerequisite:** the `sc-env` conda/mamba environment (Python, scanpy, scvi-tools, jupyterlab) must already exist — see the earlier PBMC3k notebook for full first-time terminal setup instructions. If you already have `sc-env`, just `conda activate sc-env` and `jupyter lab`.


---
## Phase 0: Download the Data (run in Terminal, NOT in this notebook)

Run this once. Skip if you already have the `data/` folder populated with all 7 samples.

```bash
cd ~/single_cell_project
mkdir -p data/ALL_sample data/ALL_sample2 data/ALL_sample3 data/ALL_sample4 \
         data/healthy_sample data/healthy_sample2 data/healthy_sample3

# --- ETV6-RUNX1_1 (GSM3872434) ---
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872434&format=file&file=GSM3872434%5FETV6%2DRUNX1%5F1%2Ebarcodes%2Etsv%2Egz" -o data/ALL_sample/barcodes.tsv.gz
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872434&format=file&file=GSM3872434%5FETV6%2DRUNX1%5F1%2Egenes%2Etsv%2Egz" -o data/ALL_sample/genes.tsv.gz
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872434&format=file&file=GSM3872434%5FETV6%2DRUNX1%5F1%2Ematrix%2Emtx%2Egz" -o data/ALL_sample/matrix.mtx.gz

# --- ETV6-RUNX1_2 (GSM3872435) ---
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872435&format=file&file=GSM3872435%5FETV6%2DRUNX1%5F2%2Ebarcodes%2Etsv%2Egz" -o data/ALL_sample2/barcodes.tsv.gz
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872435&format=file&file=GSM3872435%5FETV6%2DRUNX1%5F2%2Egenes%2Etsv%2Egz" -o data/ALL_sample2/genes.tsv.gz
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872435&format=file&file=GSM3872435%5FETV6%2DRUNX1%5F2%2Ematrix%2Emtx%2Egz" -o data/ALL_sample2/matrix.mtx.gz

# --- ETV6-RUNX1_3 (GSM3872436) ---
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872436&format=file&file=GSM3872436%5FETV6%2DRUNX1%5F3%2Ebarcodes%2Etsv%2Egz" -o data/ALL_sample3/barcodes.tsv.gz
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872436&format=file&file=GSM3872436%5FETV6%2DRUNX1%5F3%2Egenes%2Etsv%2Egz" -o data/ALL_sample3/genes.tsv.gz
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872436&format=file&file=GSM3872436%5FETV6%2DRUNX1%5F3%2Ematrix%2Emtx%2Egz" -o data/ALL_sample3/matrix.mtx.gz

# --- ETV6-RUNX1_4 (GSM3872437) ---
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872437&format=file&file=GSM3872437%5FETV6%2DRUNX1%5F4%2Ebarcodes%2Etsv%2Egz" -o data/ALL_sample4/barcodes.tsv.gz
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872437&format=file&file=GSM3872437%5FETV6%2DRUNX1%5F4%2Egenes%2Etsv%2Egz" -o data/ALL_sample4/genes.tsv.gz
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872437&format=file&file=GSM3872437%5FETV6%2DRUNX1%5F4%2Ematrix%2Emtx%2Egz" -o data/ALL_sample4/matrix.mtx.gz

# --- PBMMC_1 (GSM3872442) ---
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872442&format=file&file=GSM3872442%5FPBMMC%5F1%2Ebarcodes%2Etsv%2Egz" -o data/healthy_sample/barcodes.tsv.gz
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872442&format=file&file=GSM3872442%5FPBMMC%5F1%2Egenes%2Etsv%2Egz" -o data/healthy_sample/genes.tsv.gz
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872442&format=file&file=GSM3872442%5FPBMMC%5F1%2Ematrix%2Emtx%2Egz" -o data/healthy_sample/matrix.mtx.gz

# --- PBMMC_2 (GSM3872443) ---
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872443&format=file&file=GSM3872443%5FPBMMC%5F2%2Ebarcodes%2Etsv%2Egz" -o data/healthy_sample2/barcodes.tsv.gz
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872443&format=file&file=GSM3872443%5FPBMMC%5F2%2Egenes%2Etsv%2Egz" -o data/healthy_sample2/genes.tsv.gz
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872443&format=file&file=GSM3872443%5FPBMMC%5F2%2Ematrix%2Emtx%2Egz" -o data/healthy_sample2/matrix.mtx.gz

# --- PBMMC_3 (GSM3872444) ---
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872444&format=file&file=GSM3872444%5FPBMMC%5F3%2Ebarcodes%2Etsv%2Egz" -o data/healthy_sample3/barcodes.tsv.gz
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872444&format=file&file=GSM3872444%5FPBMMC%5F3%2Egenes%2Etsv%2Egz" -o data/healthy_sample3/genes.tsv.gz
curl -L "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM3872444&format=file&file=GSM3872444%5FPBMMC%5F3%2Ematrix%2Emtx%2Egz" -o data/healthy_sample3/matrix.mtx.gz

# Verify - every matrix.mtx.gz should be well over 1KB
ls -la data/ALL_sample*/ data/healthy_sample*/
```

**Important format note:** these GEO files use the older 2-column `genes.tsv.gz` format (gene_id, gene_symbol only — no 3rd "feature_type" column). Do **not** rename `genes.tsv.gz` to `features.tsv.gz` — scanpy's `read_10x_mtx()` auto-detection assumes a 3-column format for that filename and will error (`KeyError: 2`). The manual loader function in Step 1 below sidesteps this entirely and works regardless of naming.

---


## Step 1 — Imports and a Robust 10x Loader Function

Standard imports, plus a manual loader (`load_10x_legacy`) that reads the older-format 10x files directly via `scipy`/`anndata`, bypassing scanpy's `read_10x_mtx()` auto-detection issues with this particular dataset's file naming/format.


In [ ]:
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io
import anndata as ad


def load_10x_legacy(path, sample_name, condition):
    """Manually load a legacy-format 10x mtx directory (2-column genes.tsv.gz, no features.tsv)."""
    mat = scipy.io.mmread(f"{path}/matrix.mtx.gz").T.tocsr()
    genes = pd.read_csv(f"{path}/genes.tsv.gz", header=None, sep="\t")
    barcodes = pd.read_csv(f"{path}/barcodes.tsv.gz", header=None, sep="\t")

    adata = ad.AnnData(X=mat)
    adata.var_names = genes[1].astype(str).values   # gene symbols
    adata.var["gene_ids"] = genes[0].astype(str).values
    adata.obs_names = barcodes[0].astype(str).values
    adata.var_names_make_unique()

    adata.obs["sample"] = sample_name
    adata.obs["condition"] = condition
    return adata


print("Setup complete.")

---
# Part A — Single-Patient Pilot (ETV6-RUNX1_1 vs. PBMMC_1)

A quick end-to-end run on one leukemia sample and one healthy sample, to validate the pipeline before scaling up. **This section is optional to rerun** if you just want the full multi-patient result — skip to Part B if so. Results here carry an important caveat: with n=1 vs. n=1, any "significant" gene is a candidate finding, not a replicated one (see Part B for the properly replicated version).


## Step A1 — Load and Merge the Two Samples

In [ ]:
adata_all = load_10x_legacy("data/ALL_sample", "ETV6-RUNX1_1", "Leukemia")
adata_healthy = load_10x_legacy("data/healthy_sample", "PBMMC_1", "Healthy")

print(f"Leukemia sample: {adata_all.n_obs} cells, {adata_all.n_vars} genes")
print(f"Healthy sample: {adata_healthy.n_obs} cells, {adata_healthy.n_vars} genes")

adata = ad.concat([adata_all, adata_healthy], join="outer", label="batch",
                   keys=["ALL", "Healthy"], index_unique="-")
adata.obs_names_make_unique()

print(f"\nCombined dataset: {adata.n_obs} cells, {adata.n_vars} genes")
print(adata.obs["condition"].value_counts())

## Step A2 — Quality Control

QC metrics split by condition, then filtered using thresholds matched to published methodology for this exact dataset (200+ genes, <5000 genes, <10% mitochondrial — leukemia samples tend to have higher mito% than the 5% PBMC3k threshold would allow).

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, inplace=True, log1p=False)

sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
             jitter=0.4, multi_panel=True, groupby="condition")

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[(adata.obs["n_genes_by_counts"] < 5000) & (adata.obs["pct_counts_mt"] < 10), :].copy()

print(f"After QC filtering: {adata.n_obs} cells remain.")
print(adata.obs["condition"].value_counts())

## Step A3 — Feature Selection and Deep Learning (scVI)

Flags highly variable genes (batch-aware), then trains an scVI variational autoencoder with `batch_key="batch"` so it explicitly corrects for the ALL-vs-Healthy sample origin as a technical effect, letting real biology (cell type / disease status) drive the clustering.

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3", layer="counts", batch_key="batch")
print(f"{adata.var['highly_variable'].sum()} genes flagged as highly variable.")

In [ ]:
adata_hvg = adata[:, adata.var["highly_variable"]].copy()

scvi.model.SCVI.setup_anndata(adata_hvg, layer="counts", batch_key="batch")
vae_model = scvi.model.SCVI(adata_hvg)
vae_model.train()

## Step A4 — Clustering and UMAP

In [ ]:
adata.obsm["X_scvi"] = vae_model.get_latent_representation()

sc.pp.neighbors(adata, use_rep="X_scvi")
sc.tl.leiden(adata, resolution=0.5)
sc.tl.umap(adata)

sc.pl.umap(adata, color=["leiden", "condition"], title=["Cell Clusters (scVI-corrected)", "Condition"])

## Step A5 — Cluster Composition and Marker Genes

Quantifies what fraction of each cluster comes from the leukemia vs. healthy sample, then checks B-ALL-relevant marker genes (B-lineage/blast markers, erythroid markers, immune cell markers) to identify cluster identities.

In [ ]:
composition = pd.crosstab(adata.obs["leiden"], adata.obs["condition"])
composition["pct_leukemia"] = (composition["Leukemia"] / (composition["Leukemia"] + composition["Healthy"]) * 100).round(1)
print(composition)

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

In [ ]:
b_all_markers = ["CD19", "CD34", "MME", "VPREB1", "DNTT", "CD79A", "CD79B"]
erythroid_markers = ["HBB", "HBA1", "GYPA", "ALAS2"]
immune_markers = ["CD3D", "CD3E", "NKG7", "LYZ", "CD14"]

all_check = b_all_markers + erythroid_markers + immune_markers
available = [g for g in all_check if g in adata.var_names]
print("Available markers:", available)

sc.pl.dotplot(adata, available, groupby="leiden")

## Step A6 — Annotate Clusters

Based on the marker gene patterns above. **Adjust this mapping if your own rerun produces different cluster numbers/patterns** — always cross-check marker genes before trusting an annotation.

In [ ]:
cluster_labels = {
    "0": "Leukemic blasts",
    "1": "Leukemic blasts",
    "2": "B-cell precursors (mixed)",
    "3": "T cells",
    "4": "Monocytes",
    "5": "T cells",
    "6": "Proliferating blasts",
    "7": "Mature B cells",
    "8": "Erythroid cells"
}

adata.obs["cell_type"] = adata.obs["leiden"].map(cluster_labels)
sc.pl.umap(adata, color="cell_type", title="Annotated Cell Types (Single-Patient Pilot)")

## Step A7 — Volcano Plot: Leukemic Blasts vs. Normal Cells

Combines both blast-related labels, then compares against all normal cell types combined. **Caveat: n=1 patient vs. n=1 donor** — treat results as candidate findings pending replication (see Part B).

In [ ]:
adata.obs["blast_status"] = adata.obs["cell_type"].apply(
    lambda x: "Leukemic blasts" if x in ["Leukemic blasts", "Proliferating blasts"] else "Normal cells"
)
print(adata.obs["blast_status"].value_counts())

sc.tl.rank_genes_groups(adata, "blast_status", groups=["Leukemic blasts"], reference="Normal cells", method="wilcoxon")
result = adata.uns["rank_genes_groups"]

df = pd.DataFrame({
    "gene": result["names"]["Leukemic blasts"],
    "log2fc": result["logfoldchanges"]["Leukemic blasts"],
    "padj": result["pvals_adj"]["Leukemic blasts"]
})
df["neg_log10_padj"] = -np.log10(df["padj"].clip(lower=1e-300))
sig = (df["padj"] < 0.05) & (df["log2fc"].abs() > 1)

plt.figure(figsize=(7, 6))
plt.scatter(df.loc[~sig, "log2fc"], df.loc[~sig, "neg_log10_padj"], s=5, color="grey", alpha=0.5)
plt.scatter(df.loc[sig, "log2fc"], df.loc[sig, "neg_log10_padj"], s=5, color="red")
plt.axhline(-np.log10(0.05), linestyle="--", color="black", linewidth=0.8)
plt.axvline(1, linestyle="--", color="black", linewidth=0.8)
plt.axvline(-1, linestyle="--", color="black", linewidth=0.8)
plt.xlabel("log2 fold change")
plt.ylabel("-log10(adjusted p-value)")
plt.title("Leukemic Blasts vs. Normal Cells (Single-Patient Pilot)")

top_up = df[sig & (df["log2fc"] > 0)].sort_values("padj").head(10)
top_down = df[sig & (df["log2fc"] < 0)].sort_values("padj").head(10)
top_labels = pd.concat([top_up, top_down])
for _, row in top_labels.iterrows():
    plt.annotate(row["gene"], (row["log2fc"], row["neg_log10_padj"]), fontsize=7, ha="center", va="bottom")

plt.show()

In [ ]:
top_up = df[sig & (df["log2fc"] > 0)].sort_values("padj").head(25)
top_down = df[sig & (df["log2fc"] < 0)].sort_values("padj").head(25)

print("Top 25 UPregulated genes in Leukemic Blasts (single-patient pilot):")
print(top_up[["gene", "log2fc", "padj"]].to_string(index=False))
print("\nTop 25 DOWNregulated genes in Leukemic Blasts (single-patient pilot):")
print(top_down[["gene", "log2fc", "padj"]].to_string(index=False))

---
# Part B — Multi-Patient Analysis (4 Leukemia Patients vs. 3 Healthy Donors)

The statistically meaningful version of this analysis: pools all 7 available samples so that "significant" genes reflect a pattern replicated across multiple independent patients, not one individual's biology. This is the version whose results should be trusted/reported.


## Step B1 — Load All 7 Samples and Merge

In [ ]:
sample_info = [
    ("data/ALL_sample",   "ETV6-RUNX1_1", "Leukemia"),
    ("data/ALL_sample2",  "ETV6-RUNX1_2", "Leukemia"),
    ("data/ALL_sample3",  "ETV6-RUNX1_3", "Leukemia"),
    ("data/ALL_sample4",  "ETV6-RUNX1_4", "Leukemia"),
    ("data/healthy_sample",  "PBMMC_1", "Healthy"),
    ("data/healthy_sample2", "PBMMC_2", "Healthy"),
    ("data/healthy_sample3", "PBMMC_3", "Healthy"),
]

adatas = []
for path, sample_name, condition in sample_info:
    a = load_10x_legacy(path, sample_name, condition)
    print(f"{sample_name}: {a.n_obs} cells, {a.n_vars} genes")
    adatas.append(a)

adata = ad.concat(adatas, join="outer", label="batch",
                   keys=[s[1] for s in sample_info], index_unique="-")
adata.obs_names_make_unique()

print(f"\nCombined dataset: {adata.n_obs} cells, {adata.n_vars} genes")
print(adata.obs["condition"].value_counts())
print(adata.obs["sample"].value_counts())

## Step B2 — Quality Control (per-sample)

Same thresholds as Part A (200-5000 genes, <10% mito), checked per individual sample this time to catch any one sample behaving very differently from the rest — a common real-world issue with multi-patient data.

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, inplace=True, log1p=False)

sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
             jitter=0.4, multi_panel=True, groupby="sample", rotation=45)

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[(adata.obs["n_genes_by_counts"] < 5000) & (adata.obs["pct_counts_mt"] < 10), :].copy()

print(f"After QC filtering: {adata.n_obs} cells remain.")
print(adata.obs["condition"].value_counts())
print(adata.obs["sample"].value_counts())

## Step B3 — Feature Selection and Deep Learning (scVI)

`batch_key="sample"` now spans all 7 samples (not just 2) — scVI needs to correct for 7 technical sources this time. Training on ~24,000 cells takes noticeably longer than Part A (roughly 10-15 minutes on a Mac CPU).

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3", layer="counts", batch_key="sample")
print(f"{adata.var['highly_variable'].sum()} genes flagged as highly variable.")

In [ ]:
adata_hvg = adata[:, adata.var["highly_variable"]].copy()

scvi.model.SCVI.setup_anndata(adata_hvg, layer="counts", batch_key="sample")
vae_model = scvi.model.SCVI(adata_hvg)
vae_model.train()

## Step B4 — Clustering, UMAP, and Integration Diagnostics

Three views of the same UMAP: by cluster, by condition (Leukemia/Healthy), and by individual sample. The by-sample view is the key integration diagnostic — you want to see multiple different leukemia/healthy samples sharing space within a cluster (real batch correction), not each sample forming its own separate island (failed correction).

In [ ]:
adata.obsm["X_scvi"] = vae_model.get_latent_representation()

sc.pp.neighbors(adata, use_rep="X_scvi")
sc.tl.leiden(adata, resolution=0.5)
sc.tl.umap(adata)

sc.pl.umap(adata, color=["leiden", "condition"], title=["Cell Clusters (scVI-corrected)", "Condition"])
sc.pl.umap(adata, color="sample", title="By Individual Sample")

## Step B5 — Cluster Composition and Marker Genes

Two composition checks: cluster-vs-condition (same as Part A), and cluster-vs-individual-sample — the latter flags whether any "leukemia-dominant" cluster is actually just one patient's private population (a red flag) versus genuinely shared across multiple patients (real biology).

In [ ]:
composition = pd.crosstab(adata.obs["leiden"], adata.obs["condition"])
composition["pct_leukemia"] = (composition["Leukemia"] / (composition["Leukemia"] + composition["Healthy"]) * 100).round(1)
print(composition)

print("\n--- Per-sample breakdown per cluster ---")
sample_composition = pd.crosstab(adata.obs["leiden"], adata.obs["sample"])
print(sample_composition)

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

b_all_markers = ["CD19", "CD34", "MME", "VPREB1", "DNTT", "CD79A", "CD79B"]
erythroid_markers = ["HBB", "HBA1", "GYPA", "ALAS2"]
immune_markers = ["CD3D", "CD3E", "NKG7", "LYZ", "CD14"]
mature_b_markers = ["IGHM", "IGKC"]

all_check = b_all_markers + erythroid_markers + immune_markers + mature_b_markers
available = [g for g in all_check if g in adata.var_names]

sc.pl.dotplot(adata, available, groupby="leiden")

**Important:** always cross-check any dot-plot-based annotation against the actual top-ranked marker genes per cluster before finalizing labels — a dot plot only shows genes you already chose to look for, and can miss a cluster's real identity if it isn't on that list (this caught a real mislabeling during development of this notebook: two clusters that looked like "T/NK cells" by proximity on the UMAP turned out, on closer inspection, to be a ribosomal-high progenitor population and a monocyte population respectively).

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=8, sharey=False)

## Step B6 — Annotate Clusters

Based on marker genes confirmed above. **Adjust this mapping to match your own rerun's cluster numbers and marker patterns** — cluster numbering from Leiden can vary between runs.

In [ ]:
cluster_labels = {
    "0": "Leukemic blasts",
    "1": "Precursor/progenitor cells (uncertain)",
    "2": "Leukemic blasts",
    "3": "B-cell precursors (mixed)",
    "4": "Erythroid cells",
    "5": "Monocytes",
    "6": "Erythroid cells",
    "7": "Monocytes",
    "8": "Mature B cells",
    "9": "T/NK cells",
    "10": "Erythroid cells",
    "11": "Mature B cells"
}

adata.obs["cell_type"] = adata.obs["leiden"].map(cluster_labels)
sc.pl.umap(adata, color="cell_type", title="Annotated Cell Types (Multi-Patient)")

## Step B7 — Volcano Plot: Leukemic Blasts vs. Normal Cells (Multi-Patient)

The key result of this notebook — properly replicated across 4 leukemia patients and 3 healthy donors. Always check the per-sample crosstab below before trusting the volcano plot: the "Leukemic blasts" group should draw substantially from multiple different patients, not be dominated by one.

In [ ]:
adata.obs["blast_status"] = adata.obs["cell_type"].apply(
    lambda x: "Leukemic blasts" if x == "Leukemic blasts" else "Normal cells"
)
print(adata.obs["blast_status"].value_counts())
print(pd.crosstab(adata.obs["blast_status"], adata.obs["sample"]))

In [ ]:
sc.tl.rank_genes_groups(adata, "blast_status", groups=["Leukemic blasts"], reference="Normal cells", method="wilcoxon")
result = adata.uns["rank_genes_groups"]

df = pd.DataFrame({
    "gene": result["names"]["Leukemic blasts"],
    "log2fc": result["logfoldchanges"]["Leukemic blasts"],
    "padj": result["pvals_adj"]["Leukemic blasts"]
})
df["neg_log10_padj"] = -np.log10(df["padj"].clip(lower=1e-300))
sig = (df["padj"] < 0.05) & (df["log2fc"].abs() > 1)

plt.figure(figsize=(7, 6))
plt.scatter(df.loc[~sig, "log2fc"], df.loc[~sig, "neg_log10_padj"], s=5, color="grey", alpha=0.5)
plt.scatter(df.loc[sig, "log2fc"], df.loc[sig, "neg_log10_padj"], s=5, color="red")
plt.axhline(-np.log10(0.05), linestyle="--", color="black", linewidth=0.8)
plt.axvline(1, linestyle="--", color="black", linewidth=0.8)
plt.axvline(-1, linestyle="--", color="black", linewidth=0.8)
plt.xlabel("log2 fold change")
plt.ylabel("-log10(adjusted p-value)")
plt.title("Leukemic Blasts vs. Normal Cells (4 patients vs 3 healthy donors)")

top_up = df[sig & (df["log2fc"] > 0)].sort_values("padj").head(10)
top_down = df[sig & (df["log2fc"] < 0)].sort_values("padj").head(10)
top_labels = pd.concat([top_up, top_down])
for _, row in top_labels.iterrows():
    plt.annotate(row["gene"], (row["log2fc"], row["neg_log10_padj"]), fontsize=7, ha="center", va="bottom")

plt.show()

In [ ]:
top_up = df[sig & (df["log2fc"] > 0)].sort_values("padj").head(25)
top_down = df[sig & (df["log2fc"] < 0)].sort_values("padj").head(25)

print("Top 25 UPregulated genes in Leukemic Blasts (multi-patient):")
print(top_up[["gene", "log2fc", "padj"]].to_string(index=False))
print("\nTop 25 DOWNregulated genes in Leukemic Blasts (multi-patient):")
print(top_down[["gene", "log2fc", "padj"]].to_string(index=False))

## Step B8 — Supporting Plots

Dot plot, heatmap, stacked violin, and two composition breakdowns (pooled by condition, and per individual patient/donor — the latter reveals real patient-to-patient clinical heterogeneity in blast burden, which is often the most clinically interesting figure in a cancer scRNA-seq analysis).

In [ ]:
marker_genes = ["CD19", "CD34", "MME", "VPREB1", "DNTT", "CD79A", "CD79B",
                 "HBB", "HBA1", "GYPA",
                 "CD3D", "CD3E", "NKG7",
                 "LYZ", "CST3", "TYROBP",
                 "IGHM", "IGKC"]

available = [g for g in marker_genes if g in adata.var_names]
sc.pl.dotplot(adata, available, groupby="cell_type")

In [ ]:
sc.tl.rank_genes_groups(adata, "cell_type", method="wilcoxon")
sc.pl.rank_genes_groups_heatmap(adata, n_genes=5, groupby="cell_type", show_gene_labels=True)

In [ ]:
sc.pl.stacked_violin(adata, available, groupby="cell_type")

In [ ]:
comp = pd.crosstab(adata.obs["condition"], adata.obs["cell_type"], normalize="index") * 100
comp.plot(kind="bar", stacked=True, figsize=(8, 5))
plt.ylabel("% of cells")
plt.title("Cell Type Composition: Healthy vs Leukemia (Pooled)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
comp_sample = pd.crosstab(adata.obs["sample"], adata.obs["cell_type"], normalize="index") * 100
order = ["ETV6-RUNX1_1", "ETV6-RUNX1_2", "ETV6-RUNX1_3", "ETV6-RUNX1_4", "PBMMC_1", "PBMMC_2", "PBMMC_3"]
comp_sample = comp_sample.loc[order]

comp_sample.plot(kind="bar", stacked=True, figsize=(10, 5))
plt.ylabel("% of cells")
plt.title("Cell Type Composition by Individual Patient/Donor")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Step B9 — Save the Final Annotated Dataset

Reload anytime with `adata = sc.read_h5ad("all_multi_patient_analyzed.h5ad")` to continue without rerunning the ~15-minute scVI training step.

In [ ]:
adata.write("all_multi_patient_analyzed.h5ad")
print("Saved to all_multi_patient_analyzed.h5ad")

---
## Notes and Caveats

- **Single-patient (Part A) vs. multi-patient (Part B) results can genuinely disagree.** In development of this notebook, `DNTT` and `MEF2C` were top hits in Part A but did not replicate as top hits in Part B once patient-to-patient variability was accounted for — both are real, literature-documented B-ALL genes, but this specific dataset's patient variability was large enough that they weren't the top signal when pooling patients. This is exactly why the multi-patient version should be treated as the trustworthy result, and single-patient findings as hypotheses pending replication.
- **Not yet done in this notebook:** CNV-based confirmation of malignant cell identity (e.g. via `infercnvpy`), which would molecularly validate the marker-gene-based blast classification used here — a natural next extension.
- **The "Precursor/progenitor cells (uncertain)" cluster** did not resolve to a clean, confident identity from marker genes alone (dominated by ribosomal genes) — flagged honestly rather than force-labeled, and would benefit from closer inspection or a finer clustering resolution if precise identity matters for your purposes.
